# Twitter Sentiment Analysis - Exploratory Data Analysis

This notebook explores the [Sentiment140](http://help.sentiment140.com/for-students) dataset before any model is trained.

**Goals**
1. Understand the structure and quality of the data
2. Check whether the classes are balanced
3. Look at tweet lengths
4. See what the text looks like before and after preprocessing
5. Find the most frequent words in each sentiment

> Before running: prepare the dataset with `python -m src.download_data` (creates `data/sentiment140.csv`).

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Make the `src` package importable when the notebook runs from the notebooks/ folder.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import DATA_PATH, LABELS, RANDOM_STATE
from src.preprocessing import preprocess_series

sns.set_theme(style="whitegrid")

## 1. Loading the data

The processed file has two columns: `target` (0 = negative, 1 = positive) and `text` (the tweet).

In [ ]:
df = pd.read_csv(DATA_PATH)
df["sentiment"] = df["target"].map(LABELS)

print(f"Rows: {len(df):,} | Columns: {list(df.columns)}")
df.head()

In [ ]:
df.info()

## 2. Class balance

A model trained on very unbalanced data tends to favor the majority class. According to the dataset
documentation, the training set has the same number of positive and negative tweets - let's confirm it.

In [ ]:
print(df["sentiment"].value_counts())

ax = sns.countplot(data=df, x="sentiment", order=list(LABELS.values()))
ax.set_title("Tweets per sentiment")
plt.show()

## 3. Data quality

Missing values and duplicated tweets can distort both training and evaluation.

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print(f"\nDuplicated tweets: {df['text'].duplicated().sum():,}")

## 4. Tweet length

Tweets were limited to 140 characters when this dataset was collected. Does length differ between positive and negative tweets?

In [ ]:
df["n_chars"] = df["text"].str.len()
df["n_words"] = df["text"].str.split().str.len()

df[["n_chars", "n_words"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=df, x="n_chars", hue="sentiment", bins=40, ax=axes[0])
axes[0].set_title("Characters per tweet")

sns.histplot(data=df, x="n_words", hue="sentiment", discrete=True, ax=axes[1])
axes[1].set_title("Words per tweet")

plt.tight_layout()
plt.show()

## 5. Before and after preprocessing

The full dataset has 1.6 million tweets, so the next steps use a random sample to keep the notebook fast.

In [ ]:
sample = df.sample(n=min(50_000, len(df)), random_state=RANDOM_STATE).copy()
sample["clean_text"] = preprocess_series(sample["text"])

sample[["text", "clean_text"]].head(8)

In [ ]:
empty = (sample["clean_text"] == "").sum()
print(f"Tweets that became empty after preprocessing: {empty:,} ({empty / len(sample):.2%})")

## 6. Most frequent words per sentiment

In [ ]:
def top_words(texts, n=15):
    counter = Counter(word for text in texts for word in text.split())
    return pd.DataFrame(counter.most_common(n), columns=["word", "count"])


fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (label, name) in zip(axes, LABELS.items()):
    words = top_words(sample.loc[sample["target"] == label, "clean_text"])
    sns.barplot(data=words, x="count", y="word", ax=ax)
    ax.set_title(f"Top words - {name}")

plt.tight_layout()
plt.show()

## 7. Questions to answer after running the notebook

- Are the classes balanced?
- Do positive and negative tweets have similar lengths?
- How many tweets become empty after preprocessing? Is that a problem?
- Which words appear in the top list of both sentiments? What does that say about how useful raw word counts are?

The next step is turning these cleaned texts into numbers (TF-IDF) and training the classifiers - see `src/train.py`.